# City Nagel-Schreckenberg Simulation

Change `SAVE_DIR` in the first code cell to write a complete run into a different folder.

In [1]:
import os
import json
import random
from pathlib import Path

import numpy as np


def find_project_root(start=None):
    root = Path(start or Path.cwd()).resolve()

    for candidate in (root, *root.parents):
        if candidate.name == "project1" and (candidate / "src" / "nasch.py").exists():
            return candidate

    for nasch_py in root.glob("**/project1/src/nasch.py"):
        return nasch_py.parents[1]

    raise RuntimeError("Could not find project1/src/nasch.py from the current directory.")



PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

from src.nasch import (  # noqa: E402
    animate_moving_cars,
    animate_network_density,
    load_graph_network,
    plot_network,
    plot_network_density,
    plot_network_speed,
    plot_additional_result_figures,
    save_snapshots_json,
    simulate_graph_with_image,
    simulate_road,
)

# Change only this path to keep a new run's outputs separate.
SAVE_DIR = Path("results/baseline")

OUTPUTS = {
    "run": SAVE_DIR,
    "figures": SAVE_DIR / "figures",
    "analytics": SAVE_DIR / "figures" / "analytics",
    "one_road": SAVE_DIR / "figures" / "one_road",
    "density_animation": SAVE_DIR / "figures" / "traffic_density.gif",
    "moving_cars_animation": SAVE_DIR / "figures" / "moving_cars.gif",
    "config": SAVE_DIR / "config.json",
    "summary": SAVE_DIR / "summary.json",
    "snapshots": SAVE_DIR / "snapshots.json",
}

RUN_CONFIG = {
    "seed": 42,
    "graph": {
        "gexf_path": "data/erd.gexf",
        "inout_path": "data/inout.json",
        "cell_length_m": 7.0,
        "min_cells": 3,
        "k_paths": 4,
        "congestion_weight": 4.0,
        "reroute_at_junctions": True,
        "reroute_improvement_threshold": 0.05,
    },
    "one_road": {
        "iters": 500,
        "vmax": 5,
        "p": 0.3,
    },
    "initial_population": {
        "density": 0.025,
        "vmax": 5,
        "destinations_mode": "mixed",
        "city_to_city_probability": 0.8,
    },
    "simulation": {
        "iters": 1000,
        "save_image_every": 100,
        "vmax": 5,
        "p": 0.1,
        "injection_rate": 0.01,
        "max_new_cars": 2,
        "boundary_probability": 0.7,
        "boundary_sources": "in",
        "boundary_destinations": "out",
        "boundary_to_boundary_probability": 0.85,
        "city_to_city_probability": 0.8,
        "allow_u_turn": False,
        "record_snapshots": True,
    },
    "diagnostics": {
        "routing_samples": 500,
    },
    "animation": {
        "every": 10,
        "fps": 8,
        "max_frames": 120,
    },
    "analytics": {
        "congestion_threshold": 0.6,
    },
}


def jsonable(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {key: jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [jsonable(item) for item in value]
    return value


random.seed(RUN_CONFIG["seed"])
np.random.seed(RUN_CONFIG["seed"])

for output_dir in (OUTPUTS["run"], OUTPUTS["figures"], OUTPUTS["analytics"], OUTPUTS["one_road"]):
    output_dir.mkdir(parents=True, exist_ok=True)

with OUTPUTS["config"].open("w") as f:
    json.dump(
        jsonable({"project_root": PROJECT_ROOT, "outputs": OUTPUTS, "config": RUN_CONFIG}),
        f,
        indent=2,
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"Saving run to: {OUTPUTS['run'].resolve()}")

Project root: /mnt/4214D50D14D50537/LAPTOP/PhD_uni/classes/sem04/agent_based_modelling/DataDrivenAndAgentBasedModellingBMETEFTBsPAAMD-00/project1
Saving run to: /mnt/4214D50D14D50537/LAPTOP/PhD_uni/classes/sem04/agent_based_modelling/DataDrivenAndAgentBasedModellingBMETEFTBsPAAMD-00/project1/results/baseline


In [2]:
highway = [
    -1, -1, -1, -1, -1, 2, 3, -1, -1, 4,
    -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
    -1, 2, 3, -1, -1, 4, -1, -1, -1, -1, -1, -1,
] * 8

simulate_road(
    highway,
    output=str(OUTPUTS["one_road"] / "nasch.png"),
    **RUN_CONFIG["one_road"],
)

Saved results/baseline/figures/one_road/nasch.png


In [3]:
G, network = load_graph_network(**RUN_CONFIG["graph"])
network.routing_diagnostics(samples=RUN_CONFIG["diagnostics"]["routing_samples"])

plot_network(G, output=str(OUTPUTS["figures"] / "network.png"))

network.populate_random_od(**RUN_CONFIG["initial_population"])
plot_network_density(
    G,
    network,
    output=str(OUTPUTS["figures"] / "network_density_init.png"),
    use_density=True,
)

simulate_graph_with_image(
    G,
    network,
    output=str(OUTPUTS["figures"]),
    **RUN_CONFIG["simulation"],
)

plot_network_density(
    G,
    network,
    output=str(OUTPUTS["figures"] / "network_density_final.png"),
    use_density=True,
)
plot_network_speed(
    G,
    network,
    output=str(OUTPUTS["figures"] / "network_speed_final.png"),
)
save_snapshots_json(network, output=str(OUTPUTS["snapshots"]))

summary = {
    "nodes": len(network.nodes),
    "directed_edges": len(network.edges),
    "cells": sum(len(road) for road in network.roads),
    "cars_on_network": network.total_cars_on_network(),
    "spawned_cars": network.spawned_cars,
    "finished_cars": network.finished_cars,
    "failed_spawns": network.failed_spawns,
    "accepted_junction_moves": network.accepted_junction_moves,
    "blocked_junction_moves": network.blocked_junction_moves,
    "mean_speed": network.mean_speed(),
}

with OUTPUTS["summary"].open("w") as f:
    json.dump(jsonable(summary), f, indent=2)

analytics_outputs = plot_additional_result_figures(
    G,
    network,
    output_dir=str(OUTPUTS["analytics"]),
    scenario_summaries=[OUTPUTS["summary"]],
    scenario_labels=[SAVE_DIR.name],
    **RUN_CONFIG["analytics"],
)

summary["analytics_outputs"] = analytics_outputs
with OUTPUTS["summary"].open("w") as f:
    json.dump(jsonable(summary), f, indent=2)

summary

Routing diagnostics
  simulated directed edges: 1964
  reachable OD samples: 500
  unreachable OD samples: 0
Saved results/baseline/figures/network.png
Saved results/baseline/figures/network_density_init.png
Saved results/baseline/figures/network_density_100.png
Saved results/baseline/figures/network_speed_100.png
100 cars: 1031 finished: 136 mean speed: 0.49
Saved results/baseline/figures/network_density_200.png
Saved results/baseline/figures/network_speed_200.png
200 cars: 1207 finished: 160 mean speed: 0.45
Saved results/baseline/figures/network_density_300.png
Saved results/baseline/figures/network_speed_300.png
300 cars: 1392 finished: 175 mean speed: 0.32
Saved results/baseline/figures/network_density_400.png
Saved results/baseline/figures/network_speed_400.png
400 cars: 1576 finished: 191 mean speed: 0.24
Saved results/baseline/figures/network_density_500.png
Saved results/baseline/figures/network_speed_500.png
500 cars: 1762 finished: 205 mean speed: 0.2
Saved results/baseline/

{'nodes': 616,
 'directed_edges': 1964,
 'cells': 38372,
 'cars_on_network': 2686,
 'spawned_cars': 2967,
 'finished_cars': 281,
 'failed_spawns': 440,
 'accepted_junction_moves': 28179,
 'blocked_junction_moves': 593,
 'mean_speed': 0.12509307520476545,
 'analytics_outputs': {'traffic_timeseries': 'results/baseline/figures/analytics/traffic_timeseries.png',
  'fundamental_diagram': 'results/baseline/figures/analytics/fundamental_diagram.png',
  'bottleneck_map': 'results/baseline/figures/analytics/bottleneck_map.png',
  'bottleneck_ranking': 'results/baseline/figures/analytics/bottleneck_ranking.png',
  'junction_pressure_map': 'results/baseline/figures/analytics/junction_pressure_map.png',
  'od_flow_matrix': 'results/baseline/figures/analytics/od_flow_matrix.png',
  'scenario_comparison': 'results/baseline/figures/analytics/scenario_comparison.png'}}

In [4]:
animate_network_density(
    G,
    network,
    output=str(OUTPUTS["density_animation"]),
    **RUN_CONFIG["animation"],
)

animate_moving_cars(
    G,
    network,
    output=str(OUTPUTS["moving_cars_animation"]),
    **RUN_CONFIG["animation"],
)

{
    "density_animation": str(OUTPUTS["density_animation"]),
    "moving_cars_animation": str(OUTPUTS["moving_cars_animation"]),
}


Saved results/baseline/figures/traffic_density.gif
Saved results/baseline/figures/moving_cars.gif


{'density_animation': 'results/baseline/figures/traffic_density.gif',
 'moving_cars_animation': 'results/baseline/figures/moving_cars.gif'}